# Extract payout addresses (the expensive, one-time pull)

To add the second attribution method (H2) and the heuristic axis (H1-H5), you need
each block's coinbase **payout addresses** -- which live in the big `transactions`
table. This is the pull we kept deferring.

**Cost-first design.** A single recent month can scan tens of GB (the ordinals
era), so we never run blindly:

1. **Phase 1** dry-runs *every year* (free, no scan) and shows the total, so you
   approve the cost before a single byte is scanned.
2. **Phase 2** extracts year-by-year and caches each year. It's resumable -- if it
   stops, already-done years are skipped on the next run.

Then we merge the addresses with your cached tag data and attribute every block
with both methods. Same `PROJECT_ID` as before.

In [ ]:
%matplotlib inline
import os, glob
import numpy as np
import pandas as pd
from google.cloud import bigquery

PROJECT_ID = "your-project-id"   # <-- same as before
client = bigquery.Client(project=PROJECT_ID)

YEARS = range(2009, 2027)   # genesis (2009) .. current year

def address_query(year):
    return f"""
    SELECT
      block_number AS height,
      ARRAY(SELECT addr FROM UNNEST(outputs) AS o, UNNEST(o.addresses) AS addr) AS output_addresses
    FROM `bigquery-public-data.crypto_bitcoin.transactions`
    WHERE is_coinbase = TRUE
      AND block_timestamp_month >= '{year}-01-01'
      AND block_timestamp_month <  '{year + 1}-01-01'
    """

print("client ready:", client.project)

## Phase 1 — dry-run every year (free), see the total

This scans nothing. It asks BigQuery how much each year *would* cost, so you can
eyeball the per-year breakdown and the total before committing. The numbers should
**grow over time** (more transactions in later years) -- if every year shows the
same large number, time-pruning isn't working, so stop and tell me. The guard
halts you if the total is too big for one month's free tier.

In [ ]:
BUDGET_GB = 300   # pruned scan is ~60-70 GB total; this is generous headroom

est = {}
for y in YEARS:
    dry = client.query(address_query(y), bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
    est[y] = dry.total_bytes_processed / 1e9

for y in YEARS:
    print(f"  {y}: {est[y]:8.2f} GB   " + "#" * int(est[y] / 5))
total = sum(est.values())
print("-" * 30)
print(f"  TOTAL one-time scan: {total:.0f} GB  ({total/1000:.2f} TB).  Free tier = 1000 GB/month.")
assert total < BUDGET_GB, f"Total {total:.0f} GB exceeds budget -- split the heavy years into months."

## Phase 2 — extract and cache, year by year

This is the step that actually scans (the one-time cost you just approved). Each
year is saved to its own Parquet file; re-running skips years already done and
reports only the *new* scan. May take a few minutes.

In [ ]:
os.makedirs("../data/raw/addresses", exist_ok=True)
scanned = 0.0

for y in YEARS:
    path = f"../data/raw/addresses/addresses_{y}.parquet"
    if os.path.exists(path):
        print(f"  {y}: cached, skipping")
        continue
    job = client.query(address_query(y))
    df = job.to_dataframe()
    gb = job.total_bytes_processed / 1e9
    scanned += gb
    df.to_parquet(path)
    print(f"  {y}: {len(df):>7,} coinbase blocks  |  scanned {gb:6.1f} GB  (running {scanned:6.1f} GB)")
print("done")

## Combine and merge with the tag data

Concatenate the per-year address files and join them onto your cached
`raw_blocks.parquet` (which holds the coinbase tag). Every block has exactly one
coinbase transaction, so this should cover ~100% of blocks.

In [ ]:
parts = [pd.read_parquet(p) for p in sorted(glob.glob("../data/raw/addresses/addresses_*.parquet"))]
addresses = pd.concat(parts, ignore_index=True)

raw = pd.read_parquet("../data/raw/raw_blocks.parquet")
full = raw.merge(addresses, on="height", how="left")
full.to_parquet("../data/raw/raw_blocks_full.parquet")

has_addr = full["output_addresses"].apply(lambda a: isinstance(a, (list, np.ndarray)))
print(f"{len(full):,} blocks total | matched to an address record: {has_addr.mean():.1%}")

## Attribute every block with BOTH methods

Run tag matching and address matching together, and combine them with the NA-safe
confidence scheme -- all from your tested modules. This is the full two-method
attribution over the whole chain.

Reminder on conflicts: tag and address here both come from the *same* reference
list, so they rarely disagree (CONFLICT stays tiny) -- that measures within-list
consistency, not real inter-source conflict, which needs a second independent list.
What matters here is how much **extra coverage** address matching adds.

In [ ]:
try:
    from stage2_attribute.reference_loader import load_reference
    from stage2_attribute.tag_matcher import TagMatcher
    from stage2_attribute.address_matcher import AddressMatcher
    from stage2_attribute.confidence import confidence_frame
except ModuleNotFoundError:
    import sys
    sys.path.insert(0, os.path.abspath("../src"))
    from stage2_attribute.reference_loader import load_reference
    from stage2_attribute.tag_matcher import TagMatcher
    from stage2_attribute.address_matcher import AddressMatcher
    from stage2_attribute.confidence import confidence_frame

coinbase_tags, payout_addresses = load_reference("../data/raw/pools.json")
tm = TagMatcher(coinbase_tags)
am = AddressMatcher(payout_addresses)

full["tag_pool"] = full["coinbase_param"].apply(tm.match_hex)
full["addr_pool"] = full["output_addresses"].apply(
    lambda a: am.match(a) if isinstance(a, (list, np.ndarray)) else None)
pool, conf = confidence_frame(full["tag_pool"], full["addr_pool"])
full["pool"], full["confidence"] = pool, conf

full[["height", "timestamp", "tag_pool", "addr_pool", "pool", "confidence"]] \
    .to_parquet("../data/derived/attributed_full.parquet")

print(full["confidence"].value_counts().to_string())
tag_cov = full["tag_pool"].notna().mean()
addr_cov = full["addr_pool"].notna().mean()
either = (full["tag_pool"].notna() | full["addr_pool"].notna()).mean()
print(f"\ntag coverage:     {tag_cov:.1%}")
print(f"address coverage: {addr_cov:.1%}")
print(f"either method:    {either:.1%}   (tag-only was {tag_cov:.1%} -> address adds "
      f"{either - tag_cov:.1%})")

## What you have now

`data/derived/attributed_full.parquet` -- every block attributed by **both**
methods, with a confidence tier. This is the input the heuristic sweep needs.

The two coverage numbers tell the story: how much does bringing in payout addresses
extend what you can attribute beyond tags alone? (Expect a modest add -- the btccom
address list is partial and somewhat stale; a second, current list would add more,
which is the inter-list step.)

**Next -- the full sensitivity grid.** You now have what every heuristic needs:
**H1** tag only, **H2** address only, **H3** union (either), **H4** intersection
(both agree), **H5** conflict-resolution -- crossed with **U1/U2/U3**, over time.
That's the complete LR2 sensitivity analysis. Then re-ask the headline question on
the *heuristic* axis: do the 2017 trough and post-2019 re-concentration survive
when you change *how* you identify pools, not just what you do with the unknowns?

_Your observations (total scan in GB? how much did address matching add to
coverage? which tier dominates?):_

1.
2.
